In [ ]:
import math
import torch
import pandas as pd
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, confusion_matrix

if torch.backends.mps.is_available():
    device = torch.device("mps")
    device_name = "mps"
else:
    device = torch.device("cpu")
    device_name = "cpu"

print(f"Using device: {device_name}")

In [ ]:
model_name = "textattack/distilbert-base-uncased-MRPC"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name)
model.to(device)
model.eval()

print(f"Loaded model: {model_name}")
print(f"num_labels: {model.config.num_labels}")
print(f"label2id: {model.config.label2id}")

In [ ]:
dataset = load_dataset("glue", "mrpc", split="validation")

print("Dataset split: glue/mrpc validation")
print(f"Number of examples: {len(dataset)}")
print("Example row:")
print(dataset[0])

In [ ]:
sentence1_list = dataset["sentence1"]
sentence2_list = dataset["sentence2"]
labels = dataset["label"]

print("Prepared sentence-pair inputs for tokenizer.")
print(sentence1_list[0])
print(sentence2_list[0])

In [ ]:
batch_size = 64
max_length = 128

all_predictions = []
all_confidences = []
all_logits = []

with torch.no_grad():
    for start_idx in range(0, len(dataset), batch_size):
        end_idx = min(start_idx + batch_size, len(dataset))
        batch_s1 = sentence1_list[start_idx:end_idx]
        batch_s2 = sentence2_list[start_idx:end_idx]

        encoded = tokenizer(
            batch_s1,
            batch_s2,
            padding=True,
            truncation=True,
            max_length=max_length,
            return_tensors="pt"
        )
        encoded = {k: v.to(device) for k, v in encoded.items()}

        outputs = model(**encoded)
        logits = outputs.logits
        probs = torch.softmax(logits, dim=-1)
        preds = torch.argmax(probs, dim=-1)
        confs = torch.max(probs, dim=-1).values

        all_predictions.extend(preds.detach().cpu().tolist())
        all_confidences.extend(confs.detach().cpu().tolist())
        all_logits.extend(logits.detach().cpu().tolist())

print(f"Completed batched forward-pass inference for {len(all_predictions)} examples.")

In [ ]:
accuracy = accuracy_score(labels, all_predictions)
precision, recall, f1, _ = precision_recall_fscore_support(labels, all_predictions, average="binary")
cm = confusion_matrix(labels, all_predictions)
avg_confidence = sum(all_confidences) / len(all_confidences)

print("Evaluation metrics:")
print(f"Accuracy              : {accuracy:.4f}")
print(f"Precision             : {precision:.4f}")
print(f"Recall                : {recall:.4f}")
print(f"F1                    : {f1:.4f}")
print(f"Mean max confidence   : {avg_confidence:.4f}")
print("Confusion matrix:")
print(cm)

In [ ]:
id2label = model.config.id2label if model.config.id2label is not None else {0: "LABEL_0", 1: "LABEL_1"}
errors = []

for i, (true_label, pred_label, conf) in enumerate(zip(labels, all_predictions, all_confidences)):
    if true_label != pred_label:
        row = dataset[i]
        errors.append({
            "index": i,
            "sentence1": row["sentence1"],
            "sentence2": row["sentence2"],
            "true_label": true_label,
            "pred_label": pred_label,
            "true_label_name": id2label.get(true_label, str(true_label)),
            "pred_label_name": id2label.get(pred_label, str(pred_label)),
            "confidence": float(conf),
        })

errors = sorted(errors, key=lambda x: x["confidence"], reverse=True)
num_examples_to_show = min(10, len(errors))

print(f"Total errors: {len(errors)}")
print(f"Showing {num_examples_to_show} most confident mistakes")

error_table = pd.DataFrame(errors[:num_examples_to_show])
if len(error_table) > 0:
    display(error_table[["index", "true_label", "pred_label", "true_label_name", "pred_label_name", "confidence", "sentence1", "sentence2"]])
else:
    print("No mistakes found.")

In [ ]:
print("RESULT SUMMARY")
print(f"model={model_name}")
print("dataset_split=glue/mrpc validation")
print("inference_method=explicit_AutoTokenizer_AutoModelForSequenceClassification_batched_forward_pass")
print("input_format=tokenizer_sentence_pair")
print(f"device={device_name}")
print(f"batch_size={batch_size}")
print(f"max_length={max_length}")
print(f"num_examples={len(dataset)}")
print(f"accuracy={accuracy:.4f}")
print(f"precision={precision:.4f}")
print(f"recall={recall:.4f}")
print(f"f1={f1:.4f}")
print(f"mean_max_softmax_confidence={avg_confidence:.4f}")
print(f"num_errors={len(errors)}")